In [ ]:
# Load the Movies DB into a separate in-notebook sqlite connection (instructor-only)
import os, re, sqlite3
import pandas as pd

DATA_DIR = '/content/data'   # adjust if your folder lives elsewhere

mdb = sqlite3.connect(':memory:')

def mq(sql):
    '''Run a query against the Movies DB and return a DataFrame.'''
    return pd.read_sql_query(sql, mdb)

# Postgres -> SQLite tweaks applied as we read each .sql file
for f in sorted(os.listdir(DATA_DIR)):
    if not f.endswith('.sql'):
        continue
    with open(os.path.join(DATA_DIR, f)) as fh:
        sql = fh.read()
    sql = re.sub(r'DROP SCHEMA.*?CASCADE;', '', sql, flags=re.IGNORECASE | re.DOTALL)
    sql = re.sub(r'CREATE SCHEMA\s+\w+\s*;', '', sql, flags=re.IGNORECASE)
    sql = sql.replace('movies.', '')
    sql = re.sub(r'INT NOT NULL GENERATED BY DEFAULT AS IDENTITY', 'INTEGER', sql)
    sql = re.sub(r'varchar\(\d+\)', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'decimal\(\d+,\d+\)', 'REAL', sql, flags=re.IGNORECASE)
    sql = re.sub(r'BIGINT', 'INTEGER', sql)
    sql = re.sub(r'date\s+DEFAULT', 'TEXT DEFAULT', sql)
    try:
        mdb.executescript(sql)
    except Exception:
        # Some files emit harmless 'no transaction' warnings after auto-commit; ignore.
        pass

# Sanity check
print('Movies DB loaded. Row counts:')
for t in ['movie', 'genre', 'person', 'production_company', 'keyword',
          'movie_genres', 'movie_cast', 'movie_crew', 'movie_company', 'movie_keywords']:
    try:
        n = mdb.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'  {t:22s}: {n:>8,} rows')
    except Exception:
        print(f'  {t:22s}: MISSING')

Movies DB loaded. Row counts:
  movie                 :    4,803 rows
  genre                 :       20 rows
  person                :  104,842 rows
  production_company    :    5,047 rows
  keyword               :    9,794 rows
  movie_genres          :   12,160 rows
  movie_cast            :  106,257 rows
  movie_crew            :  129,581 rows
  movie_company         :   13,677 rows
  movie_keywords        :   36,162 rows


In [ ]:
#   movie
#   genre
#   person
#   production_company
#   keyword
#   movie_genres
#   movie_cast
#   movie_crew
#   movie_company
#   movie_keywords

In [ ]:
# Task 1: Rank Movies by Popularity within Each Genre

# Use the RANK() function to rank movies by their popularity within each genre. Display the genre name, movie title, and their rank based on popularity.

mq('''
SELECT g.genre_name,
       m.title,
       m.popularity,
       RANK() OVER (PARTITION BY g.genre_id ORDER BY m.popularity DESC) AS rank_in_genre
FROM movie m
JOIN movie_genres mg ON mg.movie_id = m.movie_id
JOIN genre       g  ON  g.genre_id = mg.genre_id
ORDER BY g.genre_name, rank_in_genre
''')

,genre_name,title,popularity,rank_in_genre
0,Action,Deadpool,514.569956,1
1,Action,Guardians of the Galaxy,481.098624,2
2,Action,Mad Max: Fury Road,434.278564,3
3,Action,Jurassic World,418.708552,4
4,Action,Pirates of the Caribbean: The Curse of the Bla...,271.972889,5
...,...,...,...,...
12155,Western,The Ballad of Gregorio Cortez,0.592821,78
12156,Western,Western Religion,0.589540,79
12157,Western,Doc Holliday's Revenge,0.459400,80
12158,Western,All Hat,0.137535,81


In [ ]:


# Task 2: Identify the Top 3 Movies by Revenue within Each Production Company

# Use the NTILE() function to divide the movies produced by each production company into quartiles based on revenue. Display the company name, movie title, revenue, and quartile.


mq('''
--title
SELECT pc.company_name,
       movie.title,
       movie.revenue,
      NTILE(4) OVER (PARTITION BY pc.company_id ORDER BY revenue DESC) AS revenue_quartile
FROM movie_company mc
join movie on movie.movie_id = mc.movie_id
join production_company pc on pc.company_id = mc.company_id
WHERE movie.revenue > 0
ORDER BY pc.company_name, revenue_quartile
''')



,company_name,title,revenue,revenue_quartile
0,1.85 Films,Rubber,98017,1
1,100 Bares,El secreto de sus ojos,33965843,1
2,100 Bares,Metegol,24000000,2
3,1019 Entertainment,Captive,2801508,1
4,10th Hole Productions,The Kids Are All Right,34705850,1
...,...,...,...,...
10731,uFilm,Astérix aux Jeux Olympiques,132900000,1
10732,uFilm,Zwartboek,26193068,2
10733,uFilm,Escobar: Paradise Lost,3758328,3
10734,uFilm,The Adventurer: The Curse of the Midas Box,6399,4


In [ ]:
# Task 3: Calculate the Running Total of Movie Budgets for Each Genre

# Use the SUM() function with the ROWS frame specification to calculate the running total of movie budgets within each genre. Display the genre name, movie title, budget, and running total budget.


mq('''
SELECT g.genre_name,
       m.title,
       m.budget,
       SUM(budget) OVER (
           PARTITION BY g.genre_name
           ORDER BY release_date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_budget
FROM movie m
JOIN movie_genres mg ON mg.movie_id = m.movie_id
JOIN genre       g  ON  g.genre_id = mg.genre_id
ORDER BY g.genre_name, release_date
''')

,genre_name,title,budget,running_budget
0,Action,Hell's Angels,3950000,3950000
1,Action,The Charge of the Light Brigade,1200000,5150000
2,Action,Tycoon,0,5150000
3,Action,Sands of Iwo Jima,1000000,6150000
4,Action,Annie Get Your Gun,3768785,9918785
...,...,...,...,...
12155,Western,Western Religion,0,1956453601
12156,Western,The Ridiculous 6,60000000,2016453601
12157,Western,The Hateful Eight,44000000,2060453601
12158,Western,The Revenant,135000000,2195453601


In [ ]:
mq('''
SELECT * from
 movie
 limit 5;
''')


,movie_id,title,budget,homepage,overview,popularity,release_date,revenue,runtime,movie_status,tagline,vote_average,vote_count
0,5,Four Rooms,4000000,,It's Ted the Bellhop's first night on the job....,22.876230,1995-12-09,4300000,98,Released,Twelve outrageous guests. Four scandalous requ...,6.5,530
1,11,Star Wars,11000000,http://www.starwars.com/films/star-wars-episod...,Princess Leia is captured and held hostage by ...,126.393695,1977-05-25,775398007,121,Released,"A long time ago in a galaxy far, far away...",8.1,6624
2,12,Finding Nemo,94000000,http://disney.com/finding-nemo,"Nemo, an adventurous young clownfish, is unexp...",85.688789,2003-05-30,940335536,100,Released,"There are 3.7 trillion fish in the ocean, they...",7.6,6122
3,13,Forrest Gump,55000000,,A man with a low IQ has accomplished great thi...,138.133331,1994-07-06,677945399,142,Released,"The world will never be the same, once you've ...",8.2,7927
4,14,American Beauty,15000000,http://www.dreamworks.com/ab/,"Lester Burnham, a depressed suburban father in...",80.878605,1999-09-15,356296601,122,Released,Look closer.,7.9,3313


In [ ]:
# Task 4: Identify the Most Recent Movie for Each Genre

# Use the FIRST_VALUE() function to find the most recent movie within each genre based on the release date. Display the genre name, movie title, and release date.


mq('''
SELECT DISTINCT g.genre_name,
       FIRST_VALUE(m.title) OVER (PARTITION BY g.genre_name ORDER BY m.release_date DESC) AS most_recent_movie,
       FIRST_VALUE(m.release_date) OVER (PARTITION BY g.genre_name ORDER BY m.release_date DESC) AS most_recent_release_date
FROM movie m
JOIN movie_genres mg ON mg.movie_id = m.movie_id
JOIN genre       g  ON  g.genre_id = mg.genre_id
ORDER BY g.genre_name
''')

,genre_name,most_recent_movie,most_recent_release_date
0,Action,Suicide Squad,2016-08-02
1,Adventure,Kicks,2016-09-09
2,Animation,Sausage Party,2016-07-11
3,Comedy,Growing Up Smith,2017-02-03
4,Crime,Suicide Squad,2016-08-02
5,Documentary,"To Be Frank, Sinatra at 100",2015-12-12
6,Drama,Growing Up Smith,2017-02-03
7,Family,Growing Up Smith,2017-02-03
8,Fantasy,Pete's Dragon,2016-08-10
9,Foreign,Burn,2012-11-01


In [ ]:
# Task 1: Rank Actors by Their Appearance in Movies

# Use the DENSE_RANK() function to rank actors based on the number of movies they have appeared in. Display the actor’s name and their rank.


mq('''
SELECT
    person_name,
    movie_count,
    DENSE_RANK() OVER (ORDER BY movie_count DESC) AS actor_rank
FROM (
    SELECT
        p.person_name,
        p.person_id,
        COUNT(*) AS movie_count
    FROM movie_cast mc
    JOIN person p
        ON p.person_id = mc.person_id
    GROUP BY p.person_id, p.person_name
) AS actor_counts
ORDER BY actor_rank;
''')


,person_name,movie_count,actor_rank
0,Samuel L. Jackson,67,1
1,Robert De Niro,57,2
2,Bruce Willis,51,3
3,Matt Damon,48,4
4,Morgan Freeman,46,5
...,...,...,...
54583,Sonia Rose,1,47
54584,Joey Nappo,1,47
54585,Michelle Brzenk,1,47
54586,Matthew Withers,1,47


In [ ]:
# Task 2: Identify the Top Director by Average Movie Rating

# Use a CTE and the RANK() function to find the director with the highest average movie rating. Display the director’s name and their average rating.

mq('''
WITH director_avg AS (
    SELECT
        person.person_name,
        AVG(movie.vote_average) AS avg_rating
    FROM movie
    JOIN movie_crew
        ON movie_crew.movie_id = movie.movie_id
    JOIN person
        ON person.person_id = movie_crew.person_id
    WHERE movie_crew.job = 'Director'
    GROUP BY person.person_id, person.person_name
),
ranked_directors AS (
    SELECT
        person_name,
        avg_rating,
        RANK() OVER (ORDER BY avg_rating DESC) AS rnk
    FROM director_avg
)
SELECT
    person_name,
    avg_rating
FROM ranked_directors
WHERE rnk = 1;
''')

,person_name,avg_rating
0,Gary Sinyor,10.0


In [ ]:

# Task 3: Calculate the Cumulative Revenue of Movies Acted by Each Actor

# Use the SUM() function to calculate the cumulative revenue of movies acted by each actor. Display the actor’s name and the cumulative revenue.



mq('''
SELECT
    person.person_name,
    SUM(movie.revenue) AS cumulative_revenue
FROM movie
JOIN movie_cast
    ON movie_cast.movie_id = movie.movie_id
JOIN person
    ON person.person_id = movie_cast.person_id
GROUP BY person.person_id, person.person_name
ORDER BY cumulative_revenue DESC;
''')



,person_name,cumulative_revenue
0,Stan Lee,17364063582
1,Samuel L. Jackson,14806065788
2,Frank Welker,11614837160
3,John Ratzenberger,11038044745
4,Hugo Weaving,10822190781
...,...,...
54583,Michu00e8le Akouvi Mu00fcller,0
54584,Young Fu Fan,0
54585,Manouk Tideman,0
54586,Hong Lan Hou,0


In [ ]:
# Task 4: Identify the Director Whose Movies Have the Highest Total Budget

# Use a CTE and a window function to find the director whose movies have the highest total budget. Display the director’s name and the total budget.



mq('''
WITH director_budget AS (
    SELECT
        person.person_name,
        SUM(movie.budget) AS total_budget
    FROM movie
    JOIN movie_crew
        ON movie_crew.movie_id = movie.movie_id
    JOIN person
        ON person.person_id = movie_crew.person_id
    WHERE movie_crew.job = 'Director'
    GROUP BY person.person_id, person.person_name
),
ranked_directors AS (
    SELECT
        person_name,
        total_budget,
        RANK() OVER (ORDER BY total_budget DESC) AS rnk
    FROM director_budget
)
SELECT
    person_name,
    total_budget
FROM ranked_directors
WHERE rnk = 1;
''')

,person_name,total_budget
0,Steven Spielberg,1667500000
